[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/06_graph_laplacian_and_spectral_theory/first_principles.ipynb)

# Topic 06: Graph Laplacian and Spectral Theory

## 1. First-Principles Intuition & Motivation

Put a temperature $x_v$ at every vertex of a graph and let heat flow along the edges. Physics dictates the rate: heat leaves $v$ toward each neighbour $u$ in proportion to the difference $x_v - x_u$. Summing over neighbours,

$$
\frac{d x_v}{dt} = -\sum_{u \sim v} (x_v - x_u) = -\Big( d_v x_v - \sum_{u \sim v} x_u \Big)
$$

The operator in parentheses is the **graph Laplacian** $L = D - A$, and the whole system is $\dot{x} = -Lx$ — the discrete heat equation. Replace temperatures by voltages and you get Kirchhoff's current law; replace them by displacements and you get a network of unit springs; replace them by probabilities and you get a random walk. One matrix, four physical readings.

The reason this matters for mathematics rather than only for physics is that $L$ is *built out of differences*. Its quadratic form is a sum of squared edge differences, so it measures how *rough* a signal is on the graph. Everything a graph "wants" to do — settle to equilibrium, mix, split into loosely coupled pieces — is a statement about which signals are smooth, and therefore a statement about the small eigenvalues of $L$.

### The Laplacian as a discrete second derivative

On the integer line with unit spacing, the second difference of a function is

$$
(\Delta f)(k) = f(k+1) - 2 f(k) + f(k-1) = -\big( (f(k) - f(k+1)) + (f(k) - f(k-1)) \big)
$$

which is exactly $-(Lf)(k)$ for the path graph, where vertex $k$ has neighbours $k \pm 1$. So $L$ is the graph version of $-\nabla^2$; the sign convention is chosen to make $L$ positive semidefinite, matching the fact that $-\nabla^2$ (not $\nabla^2$) is the nonnegative operator of analysis.

The dictionary is exact enough to be useful:

| Continuum | Graph |
|---|---|
| Domain $\Omega \subset \mathbb{R}^d$ | vertex set $V$ |
| Function $f : \Omega \to \mathbb{R}$ | signal $x \in \mathbb{R}^{n}$ |
| $-\nabla^2 f$ | $Lx$ |
| Dirichlet energy $\int \Vert \nabla f \Vert^2$ | $x^{\top} L x = \sum_{\{u,v\} \in E} (x_u - x_v)^2$ |
| Heat equation $\partial_t f = \nabla^2 f$ | $\dot{x} = -Lx$, $\ x(t) = e^{-tL} x(0)$ |
| Vibration modes of a drum | eigenvectors of $L$ |
| Fourier basis $e^{i \omega t}$ | eigenvectors of $L$, "graph Fourier basis" |

### Three faces of one operator

- **Energy**: $x^{\top} L x$ is the Dirichlet energy of $x$. Constant signals cost nothing; signals that jump across many edges cost a lot.
- **Diffusion**: $e^{-tL}$ is the heat kernel. Its long-time limit is the projector onto $\ker L$, so diffusion forgets everything except which component you started in.
- **Random walk**: with $P = D^{-1} A$ the transition matrix, $L_{\mathrm{rw}} = D^{-1} L = I - P$. Eigenvalues of $L_{\mathrm{rw}}$ near $0$ are slow modes: regions the walk gets stuck in.

These three views give three different proofs of most theorems below; keeping all three in mind is the fastest way to guess a true statement before proving it.

### What the spectrum buys

Write the eigenvalues of $L$ in increasing order, $0 = \lambda_1 \le \lambda_2 \le \cdots \le \lambda_n$. Then:

- the multiplicity of $0$ **counts the connected components** — connectivity becomes a rank computation;
- $\lambda_2$, the **algebraic connectivity**, quantifies how expensive it is to cut the graph in two, via the Cheeger inequality;
- the eigenvector of $\lambda_2$, the **Fiedler vector**, is the smoothest non-constant signal, and thresholding it produces a good cut (Topic 07);
- the product $\lambda_2 \lambda_3 \cdots \lambda_n$ divided by $n$ **counts spanning trees** (Matrix-Tree, promised in Topic 03);
- $\lambda_n$ controls the largest stable time step for diffusion and the conditioning of Laplacian linear systems.

Six combinatorial questions, one eigendecomposition. That efficiency is the entire argument for spectral graph theory.

## 2. Rigorous Mathematical Definitions & Theorem Statements

Throughout, $G = (V, E)$ is a finite undirected graph with $n = \vert V \vert$ vertices and $m = \vert E \vert$ edges, possibly with nonnegative edge weights $w_{uv} = w_{vu} \ge 0$ (unweighted means $w_{uv} \in \{0,1\}$). We write $d_v = \sum_{u} w_{uv}$ for the (weighted) degree, $A = (w_{uv})$ for the adjacency matrix, and $D = \mathrm{diag}(d_1, \dots, d_n)$.

**Definition (Oriented incidence matrix).** Fix an arbitrary orientation of each edge. The incidence matrix $B \in \mathbb{R}^{n \times m}$ has, for the edge $e$ oriented from $u$ to $v$, the column $B_{\cdot e} = e_u - e_v$: entry $+1$ in row $u$, $-1$ in row $v$, and $0$ elsewhere.

**Definition (Combinatorial Laplacian).** $L = D - A$, i.e.

$$
L_{uv} = \begin{cases} d_v & u = v \\ -w_{uv} & u \neq v \end{cases}
$$

equivalently $(Lx)_v = \sum_{u \sim v} w_{uv} (x_v - x_u)$. Every row sums to zero, so $L \mathbf{1} = 0$.

**Definition (Normalized Laplacians).** For a graph with no isolated vertices,

$$
L_{\mathrm{sym}} = D^{-1/2} L D^{-1/2} = I - D^{-1/2} A D^{-1/2}, \qquad L_{\mathrm{rw}} = D^{-1} L = I - P, \quad P = D^{-1} A
$$

$L_{\mathrm{sym}}$ is symmetric; $L_{\mathrm{rw}}$ is not, but $L_{\mathrm{rw}} = D^{-1/2} L_{\mathrm{sym}} D^{1/2}$, so the two are similar and share a spectrum.

**Definition (Rayleigh quotient).** For symmetric $M$ and $x \neq 0$, $R_M(x) = \dfrac{x^{\top} M x}{x^{\top} x}$.

**Definition (Algebraic connectivity, Fiedler vector).** $a(G) = \lambda_2(L)$ is the **algebraic connectivity**; any unit eigenvector for $\lambda_2$ is a **Fiedler vector**.

**Definition (Volume, cut, conductance).** For $S \subseteq V$: $\mathrm{vol}(S) = \sum_{v \in S} d_v$, $\ \mathrm{cut}(S, \bar{S}) = \sum_{u \in S, \, v \in \bar{S}} w_{uv}$, and the **conductance**

$$
h(G) = \min_{S \, : \, 0 \lt \mathrm{vol}(S) \le \frac{1}{2}\mathrm{vol}(V)} \frac{\mathrm{cut}(S, \bar{S})}{\mathrm{vol}(S)}
$$

**Definition (Spanning-tree count).** $\tau(G)$ is the number of spanning trees of $G$ (Topic 03).

**Definition (Graph Fourier transform).** With $L = U \Lambda U^{\top}$ orthonormal, $\hat{x} = U^{\top} x$ is the **graph Fourier transform**; $\lambda_k$ plays the role of squared frequency, since $x^{\top} L x = \sum_k \lambda_k \hat{x}_k^2$.

### Theorem statements

**T1 (Incidence factorization).** $L = B W B^{\top}$ where $W = \mathrm{diag}(w_e)$; unweighted, $L = B B^{\top}$. Consequently

$$
x^{\top} L x = \sum_{\{u,v\} \in E} w_{uv} (x_u - x_v)^2
$$

independently of the chosen orientation.

**T2 (Positive semidefiniteness).** $L \succeq 0$; all eigenvalues are real and nonnegative, and $\lambda_1 = 0$ with eigenvector $\mathbf{1}$.

**T3 (Kernel counts components).** $\dim \ker L$ equals the number $c$ of connected components, and $\ker L = \mathrm{span}\{\mathbf{1}_{C_1}, \dots, \mathbf{1}_{C_c}\}$. In particular $G$ is connected $\iff \lambda_2 \gt 0$.

**T4 (Variational characterization).** $\lambda_2 = \min\{ R_L(x) : x \neq 0, \ x \perp \mathbf{1} \}$, attained exactly at Fiedler vectors; more generally Courant–Fischer gives every $\lambda_k$.

**T5 (Elementary bounds).** For a connected graph on $n \ge 2$ vertices with minimum degree $\delta$, maximum degree $\Delta$, vertex connectivity $\kappa$ and edge connectivity $\kappa'$:

$$
\lambda_2 \le \kappa \le \kappa' \le \delta, \qquad \lambda_2 \le \frac{n}{n-1}\, \delta, \qquad \Delta + 1 \le \lambda_n \le n
$$

**T6 (Normalized spectrum).** All eigenvalues of $L_{\mathrm{sym}}$ lie in $[0, 2]$, and $2$ is an eigenvalue if and only if some connected component is bipartite.

**T7 (Cheeger inequality).** With $\lambda_2^{\mathrm{sym}} = \lambda_2(L_{\mathrm{sym}})$,

$$
\frac{h(G)^2}{2} \le \lambda_2^{\mathrm{sym}} \le 2 \, h(G)
$$

**T8 (Kirchhoff's Matrix-Tree theorem).** For any graph, every cofactor of $L$ equals $\tau(G)$: deleting row and column $i$,

$$
\tau(G) = \det\big(L^{(i)}\big) = \frac{1}{n} \prod_{k=2}^{n} \lambda_k
$$

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: The incidence factorization and the energy identity (T1)

**Claim.** $L = B W B^{\top}$, hence $x^{\top} L x = \sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2$.

**Proof.** The column of $B$ belonging to edge $e = (u \to v)$ is $b_e = e_u - e_v$. Therefore

$$
B W B^{\top} = \sum_{e \in E} w_e \, b_e b_e^{\top} = \sum_{\{u,v\} \in E} w_{uv} (e_u - e_v)(e_u - e_v)^{\top}
$$

Expand one summand: $(e_u - e_v)(e_u - e_v)^{\top} = e_u e_u^{\top} + e_v e_v^{\top} - e_u e_v^{\top} - e_v e_u^{\top}$. Summing over all edges, the diagonal accumulates $\sum_{u \sim v} w_{uv} = d_u$ in position $(u,u)$, and position $(u,v)$ with $u \neq v$ accumulates $-w_{uv}$. That is precisely $D - A = L$.

Reversing an edge's orientation flips the sign of $b_e$ and leaves $b_e b_e^{\top}$ unchanged, so $L$ does not depend on the orientation. Finally

$$
x^{\top} L x = \sum_{e} w_e \, x^{\top} b_e b_e^{\top} x = \sum_{e} w_e (b_e^{\top} x)^2 = \sum_{\{u,v\} \in E} w_{uv} (x_u - x_v)^2
$$

$\blacksquare$

$$
\boxed{L = B W B^{\top} \quad \Longrightarrow \quad x^{\top} L x = \sum_{\{u,v\} \in E} w_{uv} (x_u - x_v)^2}
$$

This single identity is the seed of every remaining proof in the module.

### Proof 2: Positive semidefiniteness and the kernel (T2, T3)

**Theorem.** $L \succeq 0$; moreover, for a graph with connected components $C_1, \dots, C_c$,

$$
\ker L = \mathrm{span}\{\mathbf{1}_{C_1}, \dots, \mathbf{1}_{C_c}\}, \qquad \dim \ker L = c
$$

**Proof.**

*Semidefiniteness.* By Proof 1, $x^{\top} L x$ is a sum of nonnegative terms $w_{uv}(x_u - x_v)^2 \ge 0$, so $x^{\top} L x \ge 0$ for all $x$. Since $L$ is real symmetric, its eigenvalues are real, and $\lambda \Vert x \Vert^2 = x^{\top} L x \ge 0$ forces $\lambda \ge 0$. Also $L \mathbf{1} = D\mathbf{1} - A\mathbf{1} = 0$ because each row of $A$ sums to the corresponding degree, so $0$ is an eigenvalue.

*Characterizing the kernel.* For symmetric positive semidefinite $L$, $Lx = 0 \iff x^{\top} L x = 0$ (write $L = M M^{\top}$ with $M = B W^{1/2}$; then $x^{\top} L x = \Vert M^{\top} x \Vert^2 = 0 \iff M^{\top} x = 0 \implies Lx = M M^{\top} x = 0$, and the converse is immediate). Now

$$
x^{\top} L x = 0 \iff w_{uv}(x_u - x_v)^2 = 0 \ \text{ for every pair } \iff x_u = x_v \ \text{ for every edge } \{u,v\}
$$

A signal equal across every edge is constant along every path, hence constant on each connected component and free to take a different value on different components. Such $x$ are exactly the linear combinations $\sum_i \alpha_i \mathbf{1}_{C_i}$, and those indicators are linearly independent (disjoint supports). Therefore $\dim \ker L = c$.

*Consequence.* $\lambda_2 \gt 0 \iff c = 1 \iff G$ is connected. $\blacksquare$

$$
\boxed{\text{mult}(0 \text{ in } \mathrm{spec}\, L) = \#\{\text{connected components}\}}
$$

**Remark (block structure).** Ordering vertices by component makes $L$ block-diagonal, $L = L_1 \oplus \cdots \oplus L_c$, and $\mathrm{spec}(L)$ is the multiset union of the component spectra. Every statement about $L$ therefore reduces to the connected case.

### Proof 3: The variational characterization of $\lambda_2$ (T4)

**Theorem (Courant–Fischer, specialized).** For connected $G$,

$$
\lambda_2 = \min_{x \neq 0, \ x \perp \mathbf{1}} \frac{x^{\top} L x}{x^{\top} x} = \min_{x \neq 0, \ \sum_v x_v = 0} \frac{\sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2}{\sum_v x_v^2}
$$

and the minimizers are exactly the Fiedler vectors.

**Proof.** Diagonalize $L = \sum_{k=1}^{n} \lambda_k u_k u_k^{\top}$ with orthonormal $u_1 = \mathbf{1}/\sqrt{n}, u_2, \dots, u_n$. Any $x \perp \mathbf{1}$ expands as $x = \sum_{k \ge 2} c_k u_k$ with $c_k = u_k^{\top} x$, so

$$
\frac{x^{\top} L x}{x^{\top} x} = \frac{\sum_{k \ge 2} \lambda_k c_k^2}{\sum_{k \ge 2} c_k^2} \ge \lambda_2 \frac{\sum_{k \ge 2} c_k^2}{\sum_{k \ge 2} c_k^2} = \lambda_2
$$

because the quotient is a convex combination of the values $\lambda_2 \le \lambda_3 \le \cdots$. Equality holds iff $c_k = 0$ whenever $\lambda_k \gt \lambda_2$, i.e. iff $x$ lies in the $\lambda_2$-eigenspace. Choosing $x = u_2$ attains the bound. $\blacksquare$

**Reading.** Among all signals summing to zero (so: not constant, genuinely "spread out"), the Fiedler vector is the one with the least edge-difference energy per unit norm. It must therefore vary *slowly* across the graph, which forces it to be nearly constant inside densely connected regions and to change sign between them — the entire basis of spectral partitioning.

**Monotonicity.** Adding an edge $\{u,v\}$ of weight $w$ replaces $L$ by $L + w(e_u - e_v)(e_u - e_v)^{\top}$, a positive semidefinite rank-one update. By Weyl's inequality every eigenvalue weakly increases:

$$
\boxed{L' = L + w\,(e_u - e_v)(e_u - e_v)^{\top} \succeq L \quad \Longrightarrow \quad \lambda_k(L') \ge \lambda_k(L) \ \text{ for all } k}
$$

so more edges means larger algebraic connectivity — "more springs, stiffer graph".

### Proof 4: Elementary bounds on the spectrum (T5)

**(a) $\lambda_2 \le \frac{n}{n-1}\,\delta$.** Let $v$ be a vertex of minimum degree $\delta$ and test with $x = e_v - \frac{1}{n}\mathbf{1}$, which satisfies $x \perp \mathbf{1}$. Then $\Vert x \Vert^2 = 1 - \frac{1}{n}$, and since $x_u - x_{u'} = (e_v)_u - (e_v)_{u'}$, the energy only counts edges at $v$:

$$
x^{\top} L x = \sum_{\{u,u'\} \in E} \big((e_v)_u - (e_v)_{u'}\big)^2 = \deg(v) = \delta
$$

Hence $\lambda_2 \le \delta \big/ \big(1 - \tfrac{1}{n}\big) = \frac{n}{n-1}\,\delta$.

**(b) $\lambda_2 \le \frac{n}{n-1}\,\kappa'$ (edge connectivity).** Let $F$ be a minimum edge cut separating $S$ from $\bar S$, $\vert F \vert = \kappa'$. Test with $x = \frac{1}{\vert S \vert}\mathbf{1}_S - \frac{1}{\vert \bar S \vert}\mathbf{1}_{\bar S}$, which is orthogonal to $\mathbf{1}$. Only cut edges contribute to the energy:

$$
x^{\top} L x = \kappa' \Big(\frac{1}{\vert S \vert} + \frac{1}{\vert \bar S \vert}\Big)^2, \qquad \Vert x \Vert^2 = \frac{1}{\vert S \vert} + \frac{1}{\vert \bar S \vert}
$$

Dividing, $\lambda_2 \le \kappa'\big(\frac{1}{\vert S \vert} + \frac{1}{\vert \bar S \vert}\big) = \dfrac{\kappa'\, n}{\vert S \vert \, \vert \bar S \vert} \le \dfrac{n}{n-1}\,\kappa'$, using $\vert S \vert \, \vert \bar S \vert \ge n - 1$. Fiedler's sharper (and considerably harder) argument upgrades this to the clean chain $\lambda_2 \le \kappa \le \kappa'$ for non-complete graphs.

**(c) $\lambda_n \ge \Delta + 1$.** Let $v$ have degree $\Delta$ and let $H \subseteq G$ be the spanning subgraph consisting only of the $\Delta$ edges at $v$ — a star $K_{1,\Delta}$ plus $n - \Delta - 1$ isolated vertices, whose Laplacian spectrum is $0$ (multiplicity $n - \Delta$), $1$ (multiplicity $\Delta - 1$), and $\Delta + 1$. Since $G$ is obtained from $H$ by adding edges, monotonicity (Proof 3) gives

$$
\lambda_n(G) \ge \lambda_n(H) = \Delta + 1
$$

**(d) $\lambda_n \le n$.** The complete graph has $L(K_n) = nI - J$ with spectrum $0$ and $n$ (multiplicity $n-1$). Writing $L(K_n) = L(G) + L(\bar G)$ with $L(\bar G) \succeq 0$ and applying Weyl's inequality gives $\lambda_n(G) \le \lambda_n(K_n) = n$. Equality holds exactly when the complement $\bar{G}$ is disconnected.

$$
\boxed{\lambda_2 \le \kappa \le \kappa' \le \delta, \qquad \lambda_2 \le \frac{n}{n-1}\,\delta, \qquad \Delta + 1 \le \lambda_n \le n}
$$

**Sanity checks.** $K_n$: $\lambda_2 = n$ and $\delta = n-1$, so $\lambda_2 = \frac{n}{n-1}\delta$ — bound (a) is tight. $P_n$: $\delta = 1$ but $\lambda_2 = 2 - 2\cos(\pi/n) \to 0$ — bound (a) is far from tight for path-like graphs, which is exactly the case Cheeger handles.

### Proof 5: The normalized spectrum lies in $[0,2]$ (T6)

**Theorem.** Every eigenvalue $\mu$ of $L_{\mathrm{sym}} = I - D^{-1/2} A D^{-1/2}$ satisfies $0 \le \mu \le 2$, with $\mu = 2$ attained iff some connected component is bipartite.

**Proof.** Substituting $x = D^{-1/2} y$ into the energy identity gives the normalized Rayleigh quotient

$$
\frac{y^{\top} L_{\mathrm{sym}} y}{y^{\top} y} = \frac{x^{\top} L x}{x^{\top} D x} = \frac{\sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2}{\sum_v d_v x_v^2}
$$

Nonnegativity of numerator and denominator gives $\mu \ge 0$. For the upper bound, use the elementary inequality $(a-b)^2 \le 2(a^2 + b^2)$:

$$
\sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2 \le 2 \sum_{\{u,v\} \in E} w_{uv}(x_u^2 + x_v^2) = 2\sum_v d_v x_v^2
$$

since each vertex $v$ collects $\sum_{u \sim v} w_{uv} = d_v$. Hence $\mu \le 2$.

*Equality case.* $\mu = 2$ requires $(x_u - x_v)^2 = 2(x_u^2 + x_v^2)$, i.e. $(x_u + x_v)^2 = 0$, for every edge of the support. So $x_v = -x_u$ across every edge: the signs of $x$ define a proper $2$-colouring of the component, and $x$ is nonzero there, so the component is bipartite. Conversely, on a bipartite component with parts $X, Y$ set $x = +1$ on $X$, $x = -1$ on $Y$, $0$ elsewhere; then $y = D^{1/2} x$ achieves the value $2$. $\blacksquare$

$$
\boxed{0 = \mu_1 \le \cdots \le \mu_n \le 2, \qquad \mu_n = 2 \iff \text{a component is bipartite}}
$$

**Why normalize.** The unnormalized $\lambda_n$ can be as large as $n$, so unnormalized spectra of graphs with different degree scales are incomparable. $L_{\mathrm{sym}}$ lives on a fixed interval, which makes eigenvalues of different graphs, and of graphs at different sizes, directly comparable — the reason $L_{\mathrm{sym}}$ is standard in Chung's book and in GNN design.

### Proof 6: The Cheeger inequality — easy direction in full, hard direction sketched (T7)

**Theorem.** $\dfrac{h(G)^2}{2} \le \lambda_2^{\mathrm{sym}} \le 2\,h(G)$.

**Easy direction ($\lambda_2^{\mathrm{sym}} \le 2h$), complete proof.** By the substitution in Proof 5, with $D$-weighted orthogonality replacing plain orthogonality,

$$
\lambda_2^{\mathrm{sym}} = \min_{x \, : \, \sum_v d_v x_v = 0} \frac{\sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2}{\sum_v d_v x_v^2}
$$

Let $S$ attain the conductance, with $\mathrm{vol}(S) \le \frac{1}{2}\mathrm{vol}(V)$, and test with

$$
x_v = \begin{cases} \ \ \frac{1}{\mathrm{vol}(S)} & v \in S \\ -\frac{1}{\mathrm{vol}(\bar S)} & v \in \bar{S} \end{cases}
$$

Feasibility: $\sum_v d_v x_v = \frac{\mathrm{vol}(S)}{\mathrm{vol}(S)} - \frac{\mathrm{vol}(\bar S)}{\mathrm{vol}(\bar S)} = 0$. ✓

Numerator: only cut edges have $x_u \neq x_v$, and each contributes $\big(\frac{1}{\mathrm{vol}(S)} + \frac{1}{\mathrm{vol}(\bar S)}\big)^2$, giving $\mathrm{cut}(S,\bar S)\big(\frac{1}{\mathrm{vol}(S)} + \frac{1}{\mathrm{vol}(\bar S)}\big)^2$.

Denominator: $\frac{\mathrm{vol}(S)}{\mathrm{vol}(S)^2} + \frac{\mathrm{vol}(\bar S)}{\mathrm{vol}(\bar S)^2} = \frac{1}{\mathrm{vol}(S)} + \frac{1}{\mathrm{vol}(\bar S)}$.

Dividing, one factor cancels:

$$
\lambda_2^{\mathrm{sym}} \le \mathrm{cut}(S,\bar S)\Big(\frac{1}{\mathrm{vol}(S)} + \frac{1}{\mathrm{vol}(\bar S)}\Big) \le \frac{2\,\mathrm{cut}(S,\bar S)}{\mathrm{vol}(S)} = 2 h(G)
$$

using $\mathrm{vol}(\bar S) \ge \mathrm{vol}(S)$. $\blacksquare$

**Hard direction ($h \le \sqrt{2 \lambda_2^{\mathrm{sym}}}$), sketch.** Take a minimizing $x$ (the normalized Fiedler vector) and shift it so that both $\{x \gt 0\}$ and $\{x \lt 0\}$ have volume at most $\frac12 \mathrm{vol}(V)$; keep the side with smaller Rayleigh contribution and set the other to zero, obtaining $g \ge 0$ with $\mathrm{supp}(g)$ of volume at most half and $\frac{\sum w_{uv}(g_u-g_v)^2}{\sum d_v g_v^2} \le \lambda_2^{\mathrm{sym}}$. Consider the **sweep cuts** $S_t = \{v : g_v^2 \gt t\}$. Applying Cauchy–Schwarz to $\sum_{uv} w_{uv} \vert g_u^2 - g_v^2 \vert = \sum_{uv} w_{uv}\vert g_u - g_v\vert\,(g_u + g_v)$ and the coarea identity $\int_0^{\infty} \mathrm{cut}(S_t)\,dt = \sum_{uv} w_{uv}\vert g_u^2 - g_v^2 \vert$, $\ \int_0^\infty \mathrm{vol}(S_t)\,dt = \sum_v d_v g_v^2$, yields some $t$ with $\frac{\mathrm{cut}(S_t)}{\mathrm{vol}(S_t)} \le \sqrt{2\lambda_2^{\mathrm{sym}}}$. $\square$

$$
\boxed{\frac{h^2}{2} \le \lambda_2^{\mathrm{sym}} \le 2h}
$$

**Algorithmic content.** The hard direction is *constructive*: it says that thresholding the Fiedler vector and taking the best of the $n-1$ sweep cuts produces a cut of conductance at most $\sqrt{2\lambda_2^{\mathrm{sym}}}$. That is the correctness guarantee for spectral bisection, and the only general-purpose guarantee spectral clustering enjoys (Topic 07).

### Proof 7: Kirchhoff's Matrix-Tree theorem via Cauchy–Binet (T8)

**Lemma (unimodularity).** Let $B_0$ be $B$ with row $n$ deleted, and let $S \subseteq E$ with $\vert S \vert = n-1$. Then

$$
\det\big(B_0[S]\big) = \begin{cases} \pm 1 & \text{if } S \text{ is a spanning tree} \\ 0 & \text{otherwise} \end{cases}
$$

*Proof of lemma.* If $S$ is not a spanning tree it has $n-1$ edges and is disconnected, so some component $C$ misses vertex $n$; the rows of $B_0[S]$ indexed by $C$ sum to zero (every $S$-edge with an endpoint in $C$ has both endpoints in $C$), so the determinant vanishes. If $S$ *is* a spanning tree, induct on $n$: the tree has a leaf $v \neq n$ (Topic 03's leaf lemma applied after rooting at $n$), whose row of $B_0[S]$ has a single nonzero entry $\pm 1$; expand along that row and apply the hypothesis to the tree minus $v$. $\square$

**Theorem.** $\tau(G) = \det(L^{(n)})$, where $L^{(n)}$ deletes row and column $n$.

**Proof.** From $L = B B^{\top}$ (unweighted case), deleting row and column $n$ gives $L^{(n)} = B_0 B_0^{\top}$ with $B_0 \in \mathbb{R}^{(n-1) \times m}$. The **Cauchy–Binet formula** expands the determinant of such a product over all column subsets of size $n-1$:

$$
\det\big(B_0 B_0^{\top}\big) = \sum_{S \subseteq E, \ \vert S \vert = n-1} \det\big(B_0[S]\big)^2
$$

By the lemma each term is $1$ for a spanning tree and $0$ otherwise, so the sum counts spanning trees:

$$
\boxed{\tau(G) = \det\big(L^{(i)}\big) \ \text{ for every } i}
$$

(The choice of deleted index is immaterial by relabelling.) $\blacksquare$

**Weighted version.** With $L = B W B^{\top}$ the same computation gives the weighted count $\tau_w(G) = \sum_{T} \prod_{e \in T} w_e$ — the partition function of the spanning-tree model, which is what makes the theorem useful in statistical physics.

### Corollary: the spectral form of the tree count

**Claim.** For a connected graph, $\tau(G) = \frac{1}{n}\lambda_2 \lambda_3 \cdots \lambda_n$.

**Derivation.** Consider the characteristic polynomial $p(t) = \det(tI - L) = \prod_{k=1}^{n}(t - \lambda_k)$. Since $\lambda_1 = 0$,

$$
p(t) = t \prod_{k=2}^{n} (t - \lambda_k) \quad \Longrightarrow \quad [t^1]\,p(t) = \prod_{k=2}^{n} (-\lambda_k) = (-1)^{n-1}\prod_{k=2}^{n}\lambda_k
$$

On the other hand, the general expansion $\det(tI - L) = \sum_{j=0}^{n} (-1)^{j} E_j(L)\, t^{\,n-j}$, where $E_j$ is the sum of all principal $j \times j$ minors, gives $[t^1]\,p(t) = (-1)^{n-1} E_{n-1}(L)$. Each principal $(n-1) \times (n-1)$ minor is $\det(L^{(i)}) = \tau(G)$ by the Matrix-Tree theorem, and there are $n$ of them, so $E_{n-1}(L) = n\,\tau(G)$. Equating,

$$
\prod_{k=2}^{n} \lambda_k = n\,\tau(G) \quad \Longrightarrow \quad \boxed{\ \tau(G) = \frac{1}{n}\prod_{k=2}^{n}\lambda_k\ }
$$

**Check on $K_n$.** $L(K_n) = nI - J$ has spectrum $0$ and $n$ with multiplicity $n-1$, so

$$
\tau(K_n) = \frac{n^{\,n-1}}{n} = n^{\,n-2}
$$

recovering Cayley's formula (Topic 03) as a one-line corollary of spectral theory. ✓

### Spectra of standard graphs

| Graph | Laplacian eigenvalues | $\lambda_2$ | $\tau(G)$ |
|---|---|---|---|
| Complete $K_n$ | $0$, and $n$ with multiplicity $n-1$ | $n$ | $n^{n-2}$ |
| Cycle $C_n$ | $2 - 2\cos\!\big(\tfrac{2\pi k}{n}\big)$, $k = 0,\dots,n-1$ | $2 - 2\cos\tfrac{2\pi}{n} \approx \tfrac{4\pi^2}{n^2}$ | $n$ |
| Path $P_n$ | $2 - 2\cos\!\big(\tfrac{\pi k}{n}\big) = 4\sin^2\!\tfrac{\pi k}{2n}$, $k = 0,\dots,n-1$ | $\approx \tfrac{\pi^2}{n^2}$ | $1$ |
| Star $K_{1,n-1}$ | $0$, $1$ with multiplicity $n-2$, and $n$ | $1$ | $1$ |
| Complete bipartite $K_{a,b}$ | $0$, $a$ with multiplicity $b-1$, $b$ with multiplicity $a-1$, and $a+b$ | $\min(a,b)$ for $a+b \ge 3$ | $a^{b-1} b^{a-1}$ |
| Hypercube $Q_d$ | $2k$ with multiplicity $\binom{d}{k}$, $k = 0,\dots,d$ | $2$ | $\prod_{k=1}^{d}(2k)^{\binom{d}{k}} \big/ 2^{d}$ |
| Petersen graph | $0$, $2$ with multiplicity $5$, $5$ with multiplicity $4$ | $2$ | $2000$ |

**Product rule.** For the Cartesian product $G \square H$, the eigenvalues are all sums $\lambda_i(G) + \mu_j(H)$ with eigenvectors $u_i \otimes v_j$. Since $Q_d = K_2^{\square d}$ and $\mathrm{spec}(L(K_2)) = \{0, 2\}$, the hypercube spectrum $\{2k \text{ with multiplicity } \binom{d}{k}\}$ follows immediately — a good illustration of how spectral theory composes where combinatorics does not.

**Scaling reading.** Compare $\lambda_2$: expanders and complete graphs have $\lambda_2 = \Theta(n)$ (or $\Theta(1)$ normalized); paths and cycles have $\lambda_2 = \Theta(n^{-2})$. That gap between $\Theta(1)$ and $\Theta(n^{-2})$ normalized connectivity is exactly the gap between "mixes in $O(\log n)$ steps" and "mixes in $\Theta(n^2)$ steps".

## 4. Computational & Algorithmic Insights

### Which algorithm for which eigenvalue

| Task | Method | Cost | Notes |
|---|---|---|---|
| Full spectrum, small $n$ | dense symmetric QR / divide-and-conquer (LAPACK `syevd`) | $O(n^3)$ | fine to $n \approx 10^4$ |
| A few smallest $\lambda_k$ | Lanczos with shift-invert, or LOBPCG | $O(\text{iter} \cdot m)$ per solve | needs a Laplacian solve per iteration |
| A few largest $\lambda_k$ | plain Lanczos / power iteration | $O(\text{iter} \cdot m)$ | converges fast; $\lambda_n$ is well separated |
| Fiedler vector only | deflate $\mathbf{1}$, then Lanczos or LOBPCG on $L$ restricted to $\mathbf{1}^{\perp}$ | $O(\text{iter} \cdot m)$ | always project out $\mathbf{1}$ explicitly |
| Solve $Lx = b$ | preconditioned CG with a combinatorial preconditioner | near-linear in theory (Spielman–Teng) | $b$ must satisfy $\mathbf{1}^{\top} b = 0$ |
| $\tau(G)$ | Cholesky of $L^{(i)}$, then product of squared pivots | $O(n^3)$ dense | far better than enumerating trees |

**Key structural facts for implementers.** $L$ is sparse ($2m + n$ nonzeros), symmetric, and singular. Never invert it: use the **pseudoinverse** $L^{+}$ conceptually and CG in practice, always on the subspace $\mathbf{1}^{\perp}$ where $L$ is positive definite with condition number $\lambda_n / \lambda_2$.

### Numerical caveats

- **The zero eigenvalue is only numerically zero.** Expect $\vert \lambda_1 \vert \lesssim \varepsilon_{\text{mach}} \Vert L \Vert$; deciding connectivity by testing "$\lambda_2 \gt 0$" numerically is fragile — run BFS instead (Topic 02) and use $\lambda_2$ only for *quantitative* connectivity.
- **Nearly disconnected graphs are ill-conditioned.** When $\lambda_2 \ll \lambda_n$, CG converges in $O(\sqrt{\lambda_n/\lambda_2})$ iterations; that is precisely the regime where the clustering signal is strongest, so preconditioning is not optional.
- **Degenerate eigenvalues.** Symmetric graphs (hypercubes, complete bipartite) have high multiplicities; individual eigenvectors are then meaningless, only the eigenspace is well defined. Cluster labels derived from a single eigenvector of a degenerate eigenvalue are numerical noise.
- **Normalization changes the answer, not just the scale.** $L$, $L_{\mathrm{sym}}$ and $L_{\mathrm{rw}}$ give genuinely different eigenvectors on irregular graphs; only on $d$-regular graphs do they coincide up to the affine map $\mu = \lambda / d$.
- **Weights must be nonnegative.** A single negative weight destroys positive semidefiniteness and every theorem above.

### Effective resistance and the pseudoinverse

Viewing each edge as a resistor of conductance $w_{uv}$, injecting one unit of current at $u$ and extracting it at $v$ gives potentials $x = L^{+}(e_u - e_v)$, and the **effective resistance** is

$$
R_{\mathrm{eff}}(u,v) = (e_u - e_v)^{\top} L^{+} (e_u - e_v)
$$

Three facts make this the most useful derived quantity of the module:

- $R_{\mathrm{eff}}$ is a **metric** on $V$, and $\sqrt{R_{\mathrm{eff}}}$ embeds the graph isometrically into $\mathbb{R}^{n-1}$ via $L^{+/2}$.
- **Foster's theorem**: $\sum_{\{u,v\} \in E} w_{uv} R_{\mathrm{eff}}(u,v) = n - 1$ — proved in one line by noting the sum equals $\operatorname{tr}(L^{+} L) = \mathrm{rank}(L)$.
- **Random walks**: the commute time between $u$ and $v$ equals $2m \cdot R_{\mathrm{eff}}(u,v)$ (unweighted case), tying resistance directly to hitting times.
- **Spectral sparsification**: sampling edges with probability proportional to $w_{uv} R_{\mathrm{eff}}(u,v)$ produces a graph with $O(n \log n / \epsilon^2)$ edges whose Laplacian quadratic form is a $(1 \pm \epsilon)$ approximation of the original (Spielman–Srivastava) — the theoretical foundation of graph coarsening in large-scale GNN pipelines.

### Verification strategy

- **Row sums**: every row of $L$ must sum to $0$; a nonzero row sum means a bug in degree computation.
- **Trace**: $\operatorname{tr}(L) = \sum_k \lambda_k = 2m$ for unweighted graphs (each edge contributes $1$ to two degrees).
- **Frobenius identity**: $\sum_k \lambda_k^2 = \Vert L \Vert_F^2 = \sum_v d_v^2 + 2m$.
- **Kernel dimension**: compare the numerically estimated $\dim \ker L$ against a BFS/DFS component count; they must agree.
- **Spanning trees**: check $\frac{1}{n}\prod_{k \ge 2}\lambda_k$ against $\det(L^{(1)})$ computed by LU — two independent routes to the same integer, so the result must be an integer within rounding error.
- **Cheeger sandwich**: for any explicit cut $S$ with $\mathrm{vol}(S) \le \frac{1}{2}\mathrm{vol}(V)$, verify $\lambda_2^{\mathrm{sym}} \le 2\,\mathrm{cut}(S)/\mathrm{vol}(S)$; a violation means the wrong Laplacian is being diagonalized.
- **Monotonicity**: adding any edge must not decrease any $\lambda_k$ — a cheap randomized test that catches sign and orientation errors in the incidence matrix.

## 5. Real-World Physics & AI/ML Applications

### Physics and engineering

- **Heat and diffusion on networks**: $x(t) = e^{-tL}x(0)$ describes temperature, concentration, or opinion diffusion; the relaxation time is $1/\lambda_2$, so algebraic connectivity literally sets how long equilibration takes.
- **Vibrating structures**: a mass-spring network with unit masses has equations $\ddot{x} = -Lx$, so the eigenvalues $\lambda_k$ are squared natural frequencies and eigenvectors are normal modes — "hearing the shape of a graph" is the discrete Kac problem, and cospectral graphs are its counterexamples.
- **Electrical networks**: $L$ is the conductance matrix of Kirchhoff's laws; $L^{+}$ produces effective resistances, and the Matrix-Tree theorem is the combinatorial content of network reduction.
- **Synchronization (Kuramoto)**: linearizing coupled oscillators about the synchronized state gives $\dot{\theta} = -K L \theta$; the synchronization threshold is governed by $\lambda_2$, so expanders synchronize and rings do not.
- **Statistical mechanics**: the weighted Matrix-Tree theorem is the partition function of the uniform spanning-tree model, connected to loop-erased random walks and the abelian sandpile.
- **Molecular chemistry**: the Wiener index and Kirchhoff index $\mathrm{Kf}(G) = n \sum_{k \ge 2} \lambda_k^{-1}$ correlate with physical properties of hydrocarbons.

### Discrete geometry

- **Graph drawing (Tutte / Hall)**: plotting each vertex at $(u_2(v), u_3(v))$ — the second and third eigenvectors — produces the layout minimizing total squared edge length subject to a normalization, which is why spectral layouts look natural.
- **Manifold approximation**: for points sampled from a manifold with a Gaussian similarity graph, $L_{\mathrm{rw}}$ converges (with the right bandwidth scaling) to the Laplace–Beltrami operator; graph eigenvectors converge to manifold eigenfunctions (Belkin–Niyogi).

### AI and machine learning

- **Spectral clustering** (Topic 07): the bottom eigenvectors of $L$, $L_{\mathrm{sym}}$, or $L_{\mathrm{rw}}$ give the embedding in which $k$-means separates clusters; the Cheeger inequality is the quality guarantee.
- **Laplacian eigenmaps and diffusion maps**: nonlinear dimensionality reduction that embeds data using $u_2, \dots, u_{k+1}$, preserving local neighbourhoods where PCA preserves global variance.
- **Semi-supervised learning**: label propagation minimizes $x^{\top} L x$ subject to matching the observed labels, i.e. it finds the smoothest signal consistent with supervision — a Laplacian-regularized least-squares problem whose solution is a linear solve.
- **Graph neural networks** (Topic 07): the graph Fourier transform $U^{\top}x$ defines spectral convolution; the GCN propagation rule is a first-order polynomial in the normalized adjacency, and its low-pass character explains over-smoothing.
- **Graph signal processing**: denoising, interpolation, and compression of signals on networks (sensor arrays, traffic, brain connectomes) use $\lambda_k$ as a frequency axis.
- **Positional encodings for graph transformers**: Laplacian eigenvectors serve as learned-free positional features, the graph analogue of sinusoidal encodings — with sign and basis ambiguity as the practical caveat.
- **Sparsification and coarsening**: effective-resistance sampling shrinks huge graphs while preserving all quadratic forms, enabling training on graphs too large to hold in memory.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Incidence factorization, energy identity | Godsil & Royle, *Algebraic Graph Theory*, Ch. 8; Spielman, *Spectral and Algebraic Graph Theory*, Ch. 3 |
| Kernel counts components, basic bounds | Mohar (1991), "The Laplacian spectrum of graphs"; von Luxburg (2007), §3 |
| Algebraic connectivity, Fiedler vector | Fiedler (1973), *Czechoslovak Math. J.* 23; Fiedler (1975), *Czechoslovak Math. J.* 25 |
| Normalized Laplacian, spectrum in $[0,2]$ | Chung, *Spectral Graph Theory*, Ch. 1 |
| Cheeger inequality, sweep cuts | Chung, Ch. 2; Alon & Milman (1985); Spielman, Ch. 20 |
| Matrix-Tree theorem, Cauchy–Binet proof | Kirchhoff (1847); Godsil & Royle §13.2; Bollobás, *Modern Graph Theory*, §VIII.5 |
| Spectra of standard and product graphs | Brouwer & Haemers, *Spectra of Graphs*, Ch. 1 and 3 |
| Effective resistance, Foster's theorem | Doyle & Snell, *Random Walks and Electric Networks*; Klein & Randić (1993) |
| Spectral sparsification | Spielman & Srivastava (2011), *SIAM J. Comput.* 40(6) |
| Laplacian eigenmaps, manifold limit | Belkin & Niyogi (2003), *Neural Computation* 15(6) |
| Cospectral graphs, spectral non-uniqueness | van Dam & Haemers (2003), "Which graphs are determined by their spectrum?" |

**Cross-links within this repository**

- Eigenvalues, Courant–Fischer, and the spectral theorem in full generality: [`../../linear_algebra/06_eigenvalues_eigenvectors_spectral_theory/`](../../linear_algebra/06_eigenvalues_eigenvectors_spectral_theory/README.md)
- Matrix calculus and the AI-facing treatment of graph matrices (adjacency powers, PageRank, GNN layers as matrix products): [`../../linear_algebra/10_matrix_calculus_graph_and_ai_applications/`](../../linear_algebra/10_matrix_calculus_graph_and_ai_applications/README.md) — this module deliberately does *not* duplicate that material, only supplies the spectral theory it assumes.
- Spanning trees and the combinatorial side of the Matrix-Tree theorem: [`../03_trees_and_minimum_spanning_trees/`](../03_trees_and_minimum_spanning_trees/README.md)
- Connectivity, components, and BFS/DFS: [`../02_traversal_and_connectivity/`](../02_traversal_and_connectivity/README.md)
- Downstream use in clustering and graph neural networks: [`../07_spectral_clustering_and_gnn_applications/`](../07_spectral_clustering_and_gnn_applications/README.md)
- Iterative eigensolvers (Lanczos, LOBPCG) and conjugate gradients: [`../../linear_algebra/09_numerical_spectrum_algorithms/`](../../linear_algebra/09_numerical_spectrum_algorithms/README.md)